# MLI Synthetics — Kaggle Runner

Runs the Phase 1 pipeline (stage gen → song analysis → designer LLM)
on Kaggle GPU using HuggingFace `transformers` instead of Ollama.

**Hardware:** T4 (16 GB) or P100. 4-bit quantization via `bitsandbytes`
is enabled by default so Mistral-Nemo-12B fits in 16 GB.

**Input:** mount an audio dataset under `/kaggle/input/mli-audio/` with
one or more `.wav` / `.mp3` / `.flac` files.

**Output:** `/kaggle/working/mli_outputs.zip` — one folder per song
containing `song_analysis.json`, `stage_layout.{json,txt}`,
`cue_list.json`, `summary.{md,json}`.

## 1. Setup — clone repo + install deps

In [ ]:
!git clone https://github.com/thekuhldude/mli-synthetics 2>/dev/null || (cd mli-synthetics && git pull)
!cd mli-synthetics && pip install -e ".[dev]" -q
!pip install librosa soundfile pypdf -q
# transformers + accelerate are preinstalled on Kaggle; bitsandbytes is the 4-bit loader.
!pip install bitsandbytes accelerate -q

## 2. Activate the HF client

Setting `USE_HF_CLIENT=true` **before** any `mli_synthetics` import
swaps in `HFClient` (transformers + GPU) instead of the Ollama client.
Mistral-Nemo Instruct 2407 is loaded lazily on first generate.

In [ ]:
import os
os.environ["USE_HF_CLIENT"] = "true"
os.environ["HF_LOAD_IN_4BIT"] = "true"           # 4-bit quant; turn off if you have >24GB
os.environ["HF_MODEL_ID"] = "mistralai/Mistral-Nemo-Instruct-2407"
# Larger per-chunk budget so the designer doesn't time out on bigger zones
os.environ["OLLAMA_TIMEOUT_SECONDS"] = "900"

# If the model is gated on HF, set your token in Kaggle secrets and uncomment:
# from kaggle_secrets import UserSecretsClient
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 3. Import the pipeline (HF client is now wired in)

In [ ]:
import sys
from pathlib import Path

# Make the editable install discoverable regardless of cwd
REPO = Path("/kaggle/working/mli-synthetics")
if (REPO / "src").exists():
    sys.path.insert(0, str(REPO / "src"))

from mli_synthetics.llm import get_default_client
from mli_synthetics.llm.hf_client import HFClient
from mli_synthetics.pipeline.orchestrator import Phase1Pipeline
from mli_synthetics.logging_config import configure_logging

configure_logging()
client = get_default_client()
assert isinstance(client, HFClient), f"Expected HFClient, got {type(client).__name__}"
print("LLM client:", type(client).__name__, "->", client.model_id)

## 4. Pre-warm the model (optional but recommended)

First call triggers the 12B model download + load. Doing it explicitly
lets you see progress before the real pipeline starts.

In [ ]:
import asyncio

async def _warmup():
    out = await client.generate(
        model=client.model_id,
        prompt='Reply with the JSON {"ok": true}',
        system="You output only JSON.",
        temperature=0.1,
        max_tokens=32,
        json_mode=True,
    )
    print("Warmup output:", out)

asyncio.run(_warmup())

## 5. Run the pipeline on each input song

In [ ]:
import asyncio
from pathlib import Path

# Collect every audio file under the dataset mount
INPUT_DIR = Path("/kaggle/input/mli-audio")
AUDIO_EXTS = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}
songs = sorted(
    p for p in INPUT_DIR.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS
)
print(f"Found {len(songs)} audio file(s)")
for s in songs:
    print(" -", s)

OUTPUT_ROOT = Path("/kaggle/working/outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

pipeline = Phase1Pipeline()
summaries = []
for song in songs:
    out_dir = OUTPUT_ROOT / song.stem
    print(f"\n=== {song.name} -> {out_dir} ===")
    try:
        result = asyncio.run(
            pipeline.generate_show(song, output_dir=out_dir)
        )
        summaries.append(result)
        print(
            f"  duration={result['song']['duration_s']}s  "
            f"genre={result['song']['genre']}  "
            f"venue={result['stage']['venue_size']}  "
            f"cues={result['design']['n_cues']}  "
            f"timings={result['timings_s']}"
        )
    except Exception as exc:
        print(f"  FAILED: {exc}")
        summaries.append({"audio": str(song), "error": str(exc)})

## 6. Bundle outputs into a single ZIP

In [ ]:
import json
import shutil
from pathlib import Path
from IPython.display import FileLink, display

OUTPUT_ROOT = Path("/kaggle/working/outputs")

# Write a top-level manifest with every run summary
manifest = OUTPUT_ROOT / "manifest.json"
manifest.write_text(json.dumps(summaries, indent=2), encoding="utf-8")

zip_base = Path("/kaggle/working/mli_outputs")
shutil.make_archive(str(zip_base), "zip", str(OUTPUT_ROOT))
zip_path = zip_base.with_suffix(".zip")
print(f"Wrote {zip_path}  ({zip_path.stat().st_size / 1e6:.2f} MB)")

display(FileLink(str(zip_path)))